In [20]:
import os
import openai
import re
from datetime import datetime

# openai.api_key = "your_openai_api_key"

# Folder containing images
folder = r"E:/Nutstore/GPT/SVXY"
files = os.listdir(folder)



# Group files by date automatically
date_images = {}

# Match filenames like "ANYTHING_20250102_daily.png" or "20250102_daily.png"
pattern = re.compile(r"(?P<date>\d{8})_(?P<type>daily|weekly)\.png", re.IGNORECASE)

for file in files:
    match = pattern.search(file)
    if not match:
        continue
    date = match.group("date")
    chart_type = match.group("type").lower()
    date_images.setdefault(date, {})[chart_type] = os.path.join(folder, file)




Directly use the github URL for the images is more convenient for the model to access the images. So, we will create a dictionary that maps dates to URLs of the images.

In [21]:
# Your GitHub repo base (update your username/repo/branch if needed)
github_base = "https://raw.githubusercontent.com/windwine/GPTUS/main"

# Grouping by date
date_images = {}

pattern = re.compile(r"(?P<date>\d{8})_(?P<type>daily|weekly)\.png", re.IGNORECASE)

for file in files:
    match = pattern.search(file)
    if not match:
        continue
    date = match.group("date")
    chart_type = match.group("type").lower()

    # Create the GitHub raw URL
    github_url = f"{github_base}/{file}"
    date_images.setdefault(date, {})[chart_type] = github_url

In [ ]:
import os
from openai import OpenAI
import openai


# openai.api_key = "s"

client = OpenAI(api_key="")

response = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a coding assistant that talks like a pirate.",
    input="How do I check if a Python object is an instance of a class?",
)

print(response.output_text)

Ahoy, matey! To see if a Python object be an instance of a class, ye can use the `isinstance()` function, it be a trusty tool! Here be how ye'd use it:

```python
class Ship:
    pass

my_ship = Ship()

# Check if 'my_ship' be an instance of 'Ship'
if isinstance(my_ship, Ship):
    print("Aye, 'tis a ship!")
else:
    print("Nay, 'tis not a ship.")
```

This will tell ye if `my_ship` be of the type `Ship`. Use it wisely, or ye might end up in Davy Jones' locker! Arrr! 🏴‍☠️


In [ ]:
import time

predictions = {}

for idx, date in enumerate(sorted(date_images)):
    # Skip if daily or weekly chart is missing
    daily_url = date_images[date].get("daily")
    weekly_url = date_images[date].get("weekly")
    if not daily_url or not weekly_url:
        continue

    prompt = (
        "请根据OHLC蜡烛图、成交量图以及MACD判断明天涨跌趋势。\n"
        "价钱K线图蜡烛图里面绿色代表收盘价钱高于开盘价钱（阳线），另外的一种颜色代表收盘价钱低于开盘价钱（阴线）。\n"
        "给出的图片第一张为日线，第二张为周线。\n"
        "要求：\n"
        "1. 作为一位资深的行情分析师，请充分利用你对股市技术分析的能力来做出合理的判断，但不用告诉我为什么。\n"
        "2. 请直接给出判断结论，格式应该为：\n"
        "   - 明天可能上涨，概率为 0.x 。（数值为 0.1 到 1 之间，只保留小数点后一位）\n"
        "   - 明天可能下跌，概率为 -0.x 。（数值为 -0.1 到 -1 之间，只保留小数点后一位）\n"
        "   - 明天不确定或震荡，概率为 0。\n"
    )

    # Call OpenAI GPT-4o
    # response = client.chat.completions.create(
    #     model="gpt-4o",
    #     messages=[
    #         {"role": "system", "content": "你是一位金融分析师，擅长技术分析。"},
    #         {"role": "user", "content": [
    #             {"type": "text", "text": prompt},
    #             {"type": "image_url", "image_url": {"url": daily_url}},
    #             {"type": "image_url", "image_url": {"url": weekly_url}}
    #         ]}
    #     ],
    #     max_tokens=2000
    # )

    # # Store the result
    # predictions[date] = response.choices[0].message.content

    # Pause every 10 requests
    if (idx + 1) % 10 == 0:
        print(f"⏸️ Pausing after {idx + 1} requests...")
        time.sleep(20)  # increase or decrease delay as needed



KeyboardInterrupt: 

In [26]:
import time
from openai import BadRequestError

predictions = {}

for idx, date in enumerate(sorted(date_images)):
    
    if int(date) < 20250101:
        continue
    # Skip if daily or weekly chart is missing
    daily_url = date_images[date].get("daily")
    weekly_url = date_images[date].get("weekly")
    if not daily_url or not weekly_url:
        continue

    prompt = (
        "请根据OHLC蜡烛图、成交量图以及MACD判断明天涨跌趋势。\n"
        "价钱K线图蜡烛图里面绿色代表收盘价钱高于开盘价钱（阳线），另外的一种颜色代表收盘价钱低于开盘价钱（阴线）。\n"
        "给出的图片第一张为日线，第二张为周线。\n"
        "要求：\n"
        "1. 作为一位资深的行情分析师，请充分利用你对股市技术分析的能力来做出合理的判断，但不用告诉我为什么。\n"
        "2. 请直接给出判断结论，格式应该为：\n"
        "   - 明天可能上涨，概率为 0.x。\n"
        "   - 明天可能下跌，概率为 -0.x。\n"
        "   - 明天不确定或震荡，概率为 0。\n"
    )

    attempt = 0
    max_attempts = 3
    success = False

    while attempt < max_attempts and not success:
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "你是一位金融分析师，擅长技术分析。"},
                    {"role": "user", "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": daily_url}},
                        {"type": "image_url", "image_url": {"url": weekly_url}}
                    ]}
                ],
                max_tokens=2000
            )
            predictions[date] = response.choices[0].message.content
            success = True

        except BadRequestError as e:
            if "Timeout while downloading" in str(e):
                attempt += 1
                wait_time = 30
                print(f"⚠️ Timeout fetching image for {date}. Retrying in {wait_time}s (Attempt {attempt}/{max_attempts})...")
                time.sleep(wait_time)
            else:
                print(f"❌ BadRequestError on {date}: {e}")
                break
        except Exception as e:
            print(f"❌ Unexpected error on {date}: {e}")
            break

    # Pause every 10 requests
    if (idx + 1) % 1 == 0:
        print(f"⏸️ Pausing after {idx + 1} requests...")
        time.sleep(20)


⏸️ Pausing after 253 requests...
⏸️ Pausing after 254 requests...
⏸️ Pausing after 255 requests...
⏸️ Pausing after 256 requests...
⏸️ Pausing after 257 requests...
⏸️ Pausing after 258 requests...
⏸️ Pausing after 259 requests...
⏸️ Pausing after 260 requests...
⏸️ Pausing after 261 requests...
⏸️ Pausing after 262 requests...
⏸️ Pausing after 263 requests...
⏸️ Pausing after 264 requests...
⏸️ Pausing after 265 requests...
⏸️ Pausing after 266 requests...
⏸️ Pausing after 267 requests...
⏸️ Pausing after 268 requests...
⏸️ Pausing after 269 requests...
⏸️ Pausing after 270 requests...
⏸️ Pausing after 271 requests...
⏸️ Pausing after 272 requests...
⏸️ Pausing after 273 requests...
⏸️ Pausing after 274 requests...
⏸️ Pausing after 275 requests...
⏸️ Pausing after 276 requests...
⏸️ Pausing after 277 requests...
⏸️ Pausing after 278 requests...
⏸️ Pausing after 279 requests...
⏸️ Pausing after 280 requests...
⏸️ Pausing after 281 requests...
⏸️ Pausing after 282 requests...
⏸️ Pausing

In [29]:
for date in sorted(predictions):
    print("=" * 40)
    print(f"📅 Date: {date}")
    print(f"🧠 Prediction and Reasoning:\n{predictions[date]}\n")


📅 Date: 20250102
🧠 Prediction and Reasoning:
- 明天可能上涨，概率为 0.6。
- 明天可能下跌，概率为 -0.4。
- 明天不确定或震荡，概率为 0。

📅 Date: 20250103
🧠 Prediction and Reasoning:
- 明天可能上涨，概率为 0.4。
- 明天可能下跌，概率为 -0.6。
- 明天不确定或震荡，概率为 0。

📅 Date: 20250106
🧠 Prediction and Reasoning:
- 明天可能上涨，概率为 0.4。
- 明天可能下跌，概率为 -0.3。
- 明天不确定或震荡，概率为 0.3。

📅 Date: 20250107
🧠 Prediction and Reasoning:
- 明天可能上涨，概率为 0.65。
- 明天可能下跌，概率为 -0.30。
- 明天不确定或震荡，概率为 0.05。

📅 Date: 20250108
🧠 Prediction and Reasoning:
- 明天可能上涨，概率为 0.4。
- 明天可能下跌，概率为 -0.6。
- 明天不确定或震荡，概率为 0。

📅 Date: 20250110
🧠 Prediction and Reasoning:
- 明天可能上涨，概率为 0.6。
- 明天可能下跌，概率为 -0.4。
- 明天不确定或震荡，概率为 0。

📅 Date: 20250113
🧠 Prediction and Reasoning:
- 明天可能上涨，概率为 0.4。
- 明天可能下跌，概率为 -0.6。
- 明天不确定或震荡，概率为 0。

📅 Date: 20250114
🧠 Prediction and Reasoning:
- 明天可能上涨，概率为 0.4。
- 明天可能下跌，概率为 -0.5。
- 明天不确定或震荡，概率为 0.1。

📅 Date: 20250115
🧠 Prediction and Reasoning:
- 明天可能上涨，概率为 0.6。
- 明天可能下跌，概率为 -0.4。
- 明天不确定或震荡，概率为 0。

📅 Date: 20250116
🧠 Prediction and Reasoning:
- 明天可能上涨，概率为 0.6。
- 明天可能下跌，概率为 -0.4。


In [30]:
import re
import json
import pandas as pd

# === Step 1: Parse Function ===
def parse_prediction(raw_text):
    """
    Extract direction and numeric prediction from model output like:
    - 明天可能上涨，概率为 0.6。
    - 明天可能下跌，概率为 -0.4。
    - 明天不确定或震荡，概率为 0。
    """
    # Extract numeric prediction
    prob_match = re.search(r"概率为\s*([-+]?[0-9]*\.?[0-9]+)", raw_text)
    if prob_match:
        try:
            pred_y = float(prob_match.group(1))
        except ValueError:
            pred_y = 0.0
    else:
        pred_y = 0.0

    # Determine direction
    if "上涨" in raw_text:
        direction = "上涨"
    elif "下跌" in raw_text:
        direction = "下跌"
    elif "不确定" in raw_text or "震荡" in raw_text:
        direction = "震荡"
    else:
        direction = "未知"

    return {
        "direction": direction,
        "pred_y": pred_y
    }

# === Step 2: Process Predictions ===
# Example: replace `predictions` with your actual raw prediction dictionary
# predictions = {"20250828": "明天可能上涨，概率为 0.6。", ...}

structured_preds = {}

for date, raw in predictions.items():
    parsed = parse_prediction(raw)
    structured_preds[date] = {
        "raw": raw,
        "direction": parsed["direction"],
        "pred_y": parsed["pred_y"]
    }

# === Step 3: Save to JSON ===
json_path = "forecast_predictions_with_prob.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(structured_preds, f, ensure_ascii=False, indent=2)

print(f"✅ Saved to {json_path}")

# === Step 4: Read Back and Save to CSV ===
with open(json_path, "r", encoding="utf-8") as f:
    preds = json.load(f)

df = pd.DataFrame([
    {
        "date": date,
        "direction": info["direction"],
        "pred_y": info["pred_y"],
        "raw_text": info["raw"]
    }
    for date, info in preds.items()
])

csv_path = "c:/jiaqifiles/rdata/forecast_predictions_with_prob20254omini.csv"
df.to_csv(csv_path, index=False)
print(f"📄 CSV exported to {csv_path}")




✅ Saved to forecast_predictions_with_prob.json
📄 CSV exported to c:/jiaqifiles/rdata/forecast_predictions_with_prob20254omini.csv
